<a href="https://colab.research.google.com/github/Cheth-code/jugaad-ml/blob/main/Unsloth_TEXT_TO_SQl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q unsloth

In [ ]:
# Force-install the specific version that torchaudio wants
!pip install torch==2.9.0 torchaudio==2.9.0 --extra-index-url https://download.pytorch.org/whl/cu126

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! Unsloth also supports RoPE (Rotary Positinal Embedding) scaling internally.
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.1-8B-Instruct-bnb-4bit", # or choose "unsloth/Llama-3.2-1B-Instruct"
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit, # Will load the 4Bit Quantized Model
)

In [ ]:

model = FastLanguageModel.get_peft_model(
    model,
    r = 32, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32, # a higher alpha value assigns more weight to the LoRA activations
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

In [ ]:
from datasets import load_dataset
dataset = load_dataset("b-mc2/sql-create-context", split = "train")

In [ ]:
print(dataset[:5])

In [ ]:

sql_r1_prompt = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a SQL expert. First, analyze the schema and plan your query inside <think> tags. Then, provide the final SQL query in the solution block.<|eot_id|><|start_header_id|>user<|end_header_id|>
### Database Schema:
{}

### Question:
{}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
<think>
{}
</think>

**Solution:**
```sql
{}
```<|eot_id|>"""
EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
  schemas   = examples["context"]
  questions = examples["question"]
  solutions = examples["answer"]
  texts = []

  for schema, question, solution in zip(schemas, questions, solutions):
        thought = f"I need to query the table based on the question: '{question}'. Looking at the schema, I will use the relevant columns to form a SELECT statement."

        text = sql_r1_prompt.format(schema, question, thought, solution) + EOS_TOKEN
        texts.append(text)

  return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched = True)

In [ ]:

from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2, # Number of processors to use for processing the dataset
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2, # The batch size per GPU/TPU core
        gradient_accumulation_steps = 4, # Number of steps to perform befor each gradient accumulation
        warmup_steps = 10, # Few updates with low learning rate before actual training
        max_steps = 100, # Specifies the total number of training steps (batches) to run.
        learning_rate = 2e-5,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit", # Optimizer
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc for observability
    ),
)

In [ ]:
trainer_stats = trainer.train()

In [ ]:
from unsloth.chat_templates import get_chat_template

# 1. First Principles Fix: Define the SQL System Prompt clearly
sql_sys_prompt = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a SQL expert. Follow these three parsimonious rules exactly:
1. First, analyze the database schema and plan your logic inside <think> tags.
2. Provide the final SQL query in a markdown code block.
3. MANDATORY: You MUST end every SQL query with a semicolon (;). Failure to include the semicolon is an error.

### Schema:
{}<|eot_id|><|start_header_id|>user<|end_header_id|>

{}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""

# 2. Test Data: Use your specific Cloud Billing schema
test_schema = "CREATE TABLE gcp_billing (id INT, project_id TEXT, cost FLOAT, currency TEXT, usage_start TIMESTAMP,usage_end TIMESTAMP);"
test_question = "Find the top 3 services by cost for 'finance-01' in Dec 2025 and their % of total project spend."

# 3. Formatting: Use the correct variable 'sql_sys_prompt'
message = sql_sys_prompt.format(test_schema, test_question)

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)
FastLanguageModel.for_inference(model)

messages = [
    {"role": "user", "content": message},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

# 4. Optimizer Fix: Lower temperature for SQL precision
outputs = model.generate(
    input_ids = inputs,
    max_new_tokens = 1024,
    use_cache = True,
    temperature = 0.1,  # Deterministic output for SQL
    min_p = 0.1
)

input_length = inputs.shape[1]
clean_output_tokens = outputs[0][input_length:]
clean_response = tokenizer.decode(clean_output_tokens, skip_special_tokens=True)
print(clean_response.split("assistant")[-1].strip())

In [ ]:
from huggingface_hub import login
login()

In [ ]:
# The final 'Optimizer' move to make your model public
model.push_to_hub("Chethan102/Text_sql_Ai-Cloud-Billing", token = True)